# Entrenamiento YOLO26 – Detección de armas (IoT)

**Proyecto:** `C:/Users/Usuario/PycharmProjects/TRAIN_MIGUEL`
- `data/` → aquí se descarga el dataset de Roboflow
- `models/` → aquí quedan los pesos base, los entrenamientos (`runs/`) y los modelos exportados

**Modelo recomendado para IoT: `yolo26n.pt` (nano).** Es el más liviano y rápido de la familia YOLO26, pensado para CPU y dispositivos edge (Raspberry Pi, Jetson Nano, ESP32-S3 vía servidor, etc.).
Si tu dispositivo tiene GPU (Jetson Orin) o necesitas más precisión, cambia a `yolo26s.pt` en la celda de configuración.

Ejecuta las celdas en orden.

## 1. Instalación de librerías

In [ ]:
# 1) PyTorch con CUDA 12.8 (OBLIGATORIO para RTX serie 50 / Blackwell, como tu RTX 5060)
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128

# 2) Ultralytics y Roboflow
%pip install -U ultralytics roboflow

# Después de instalar: REINICIA EL KERNEL y continúa con la celda 2

## 2. Verificar entorno (GPU / CPU)

In [1]:
import torch, ultralytics
print("Ultralytics:", ultralytics.__version__)
print("PyTorch    :", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA de PyTorch:", torch.version.cuda, "| arquitecturas:", torch.cuda.get_arch_list())
    x = torch.randn(1000, 1000, device="cuda"); print("Prueba en GPU OK:", (x @ x).shape)
else:
    print("Se entrenará en CPU (será más lento). Reinstala PyTorch con cu128 (celda 1) y reinicia el kernel.")

Ultralytics: 8.4.175
PyTorch    : 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA de PyTorch: 12.8 | arquitecturas: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
Prueba en GPU OK: torch.Size([1000, 1000])


## 3. Configuración general
Cambia aquí el modelo, épocas, tamaño de imagen, etc.

- `IMGSZ = 640` es lo estándar. Para IoT muy limitado puedes entrenar/exportar con `416` o `320` (más rápido, algo menos preciso).

In [2]:
import os
from pathlib import Path

# --- Rutas del proyecto (según tu PyCharm) ---
BASE_DIR   = Path("MIGUEL_TRAIN")
DATA_DIR   = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Modelo ---
MODEL_NAME = "yolo26n.pt"      # recomendado IoT. Opciones: yolo26n / yolo26s / yolo26m / yolo26l / yolo26x
RUN_NAME   = "weapon_" + MODEL_NAME.replace(".pt", "")

# --- Hiperparámetros ---
EPOCHS   = 100
IMGSZ    = 640
PATIENCE = 20                   # early stopping si no mejora en 20 épocas
DEVICE   = 0 if torch.cuda.is_available() else "cpu"
BATCH    = 16 if torch.cuda.is_available() else 8   # si te da "CUDA out of memory", baja a 8 o 4
WORKERS  = 2                    # en Windows + Jupyter, valores bajos evitan errores; usa 0 si se cuelga

print("Datos :", DATA_DIR)
print("Modelos:", MODELS_DIR)
print("Modelo:", MODEL_NAME, "| device:", DEVICE, "| batch:", BATCH)

Datos : MIGUEL_TRAIN\data
Modelos: MIGUEL_TRAIN\models
Modelo: yolo26n.pt | device: 0 | batch: 16


## 4. Descargar dataset desde Roboflow (a la carpeta `data/`)

In [3]:
from roboflow import Roboflow

# Recomendación: guarda la API key como variable de entorno ROBOFLOW_API_KEY y no la compartas
API_KEY = os.getenv("ROBOFLOW_API_KEY", "R4fPPXiz9e2tWXD1EA7I")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("darwin-turpo-s-workspace").project("weapon-detection-revmd-kuwt4")
version = project.version(1)

DATASET_DIR = DATA_DIR / "weapon-detection-v1"
dataset = version.download("yolo26", location=str(DATASET_DIR))
# Si el formato "yolo26" diera error en tu versión de roboflow, usa: version.download("yolov11", location=str(DATASET_DIR))

DATASET_DIR = Path(dataset.location)
DATA_YAML = DATASET_DIR / "data.yaml"
print("Dataset en:", DATASET_DIR)
print("data.yaml :", DATA_YAML, "| existe:", DATA_YAML.exists())

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to MIGUEL_TRAIN\data\weapon-detection-v1 in yolo26::  37%|███▋      | 2447/6565 [00:03<00:06, 654.14it/s] 


FileNotFoundError: [Errno 2] No such file or directory: 'MIGUEL_TRAIN\\data\\weapon-detection-v1\\valid\\images\\bladeaddictknives-sword-japanese-samurai-sword-katana-high-carbon-steel-ninja-blade-white-dragon-tang-381577330715_800x600_jpg.rf.1f24d147fa6a5632f060a60a79d92417.jpg'

## 5. Corregir rutas del `data.yaml`
Roboflow suele dejar rutas relativas (`../train/images`) que fallan en Windows/PyCharm. Aquí se reescriben con la ruta absoluta.

In [ ]:
import yaml

with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = DATASET_DIR.as_posix()
split_dirs = {"train": "train", "val": "valid", "test": "test"}
for key, folder in split_dirs.items():
    if (DATASET_DIR / folder / "images").exists():
        cfg[key] = f"{folder}/images"
    else:
        cfg.pop(key, None)
        print(f"Aviso: no existe el split '{folder}'")

# Si no hay 'valid', usar 'test' como validación
if "val" not in cfg and "test" in cfg:
    cfg["val"] = cfg["test"]

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
CLASS_NAMES = cfg["names"]

## 6. Revisar el dataset

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
for folder in ["train", "valid", "test"]:
    img_dir = DATASET_DIR / folder / "images"
    lbl_dir = DATASET_DIR / folder / "labels"
    if img_dir.exists():
        n_img = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT)
        n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
        print(f"{folder:6s} -> imágenes: {n_img:5d} | etiquetas: {n_lbl:5d}")
print("Clases:", CLASS_NAMES)

In [ ]:
# Visualizar algunas imágenes de entrenamiento con sus cajas
import random, cv2
import matplotlib.pyplot as plt

imgs = [p for p in (DATASET_DIR / "train" / "images").iterdir() if p.suffix.lower() in IMG_EXT]
sample = random.sample(imgs, min(6, len(imgs)))
names = CLASS_NAMES if isinstance(CLASS_NAMES, list) else list(CLASS_NAMES.values())

plt.figure(figsize=(15, 9))
for i, p in enumerate(sample, 1):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = DATASET_DIR / "train" / "labels" / (p.stem + ".txt")
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            c, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:5])
            x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
            x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(img, names[c], (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    plt.subplot(2, 3, i); plt.imshow(img); plt.axis("off"); plt.title(p.name[:30], fontsize=8)
plt.tight_layout(); plt.show()

## 7. Entrenamiento
Los pesos base (`yolo26n.pt`) se descargan automáticamente en `models/`.
Los resultados quedan en `models/runs/<RUN_NAME>/` (gráficas, `weights/best.pt`, `weights/last.pt`).

In [ ]:
from ultralytics import YOLO

os.chdir(MODELS_DIR)            # para que el .pt base se guarde en models/
model = YOLO(MODEL_NAME)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    project=str(MODELS_DIR / "runs"),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    seed=42,
    plots=True,
    amp=True,
    cache=False,                # pon "ram" si tienes mucha RAM para acelerar
    # Aumentos útiles para cámaras IoT (luz variable, ángulos)
    hsv_v=0.5,
    degrees=5.0,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print("Entrenamiento terminado. Mejor modelo:", BEST_PT)

### (Opcional) Reanudar un entrenamiento interrumpido
Descomenta y ejecuta solo si se cortó el entrenamiento.

In [ ]:
# RUN_DIR = MODELS_DIR / "runs" / RUN_NAME
# model = YOLO(str(RUN_DIR / "weights" / "last.pt"))
# model.train(resume=True)
# BEST_PT = RUN_DIR / "weights" / "best.pt"

## 8. Curvas de entrenamiento

In [ ]:
from IPython.display import Image, display
RUN_DIR = MODELS_DIR / "runs" / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
for name in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
    p = RUN_DIR / name
    if p.exists():
        print(name); display(Image(filename=str(p), width=900))

## 9. Validación (split *valid*) y prueba (split *test*)

In [ ]:
best_model = YOLO(str(BEST_PT))

val_metrics = best_model.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, device=DEVICE, plots=True)
print(f"VALID -> mAP50: {val_metrics.box.map50:.4f} | mAP50-95: {val_metrics.box.map:.4f} | "
      f"Precision: {val_metrics.box.mp:.4f} | Recall: {val_metrics.box.mr:.4f}")

if (DATASET_DIR / "test" / "images").exists():
    test_metrics = best_model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device=DEVICE)
    print(f"TEST  -> mAP50: {test_metrics.box.map50:.4f} | mAP50-95: {test_metrics.box.map:.4f}")

## 10. Inferencia de prueba

In [ ]:
src_dir = DATASET_DIR / ("test" if (DATASET_DIR / "test" / "images").exists() else "valid") / "images"
test_imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in IMG_EXT]
sample = random.sample(test_imgs, min(6, len(test_imgs)))

preds = best_model.predict(source=[str(p) for p in sample], imgsz=IMGSZ, conf=0.35, device=DEVICE, verbose=False)

plt.figure(figsize=(15, 9))
for i, r in enumerate(preds, 1):
    plt.subplot(2, 3, i)
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    plt.axis("off"); plt.title(f"{len(r.boxes)} detecciones | {r.speed['inference']:.1f} ms", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# (Opcional) Probar con una imagen o video propio, o con la webcam (source=0)
# best_model.predict(source="C:/ruta/a/mi_video.mp4", imgsz=IMGSZ, conf=0.35, save=True, project=str(MODELS_DIR / "predicciones"))

## 11. Exportar para IoT
Elige el formato según tu dispositivo:

| Dispositivo | Formato |
|---|---|
| Raspberry Pi / ARM (CPU) | **NCNN** (el más rápido en ARM) o TFLite |
| Microcontroladores / Coral / Android | **TFLite INT8** |
| PC / servidor / multiplataforma | **ONNX** |
| CPU Intel | OpenVINO |
| Jetson (GPU NVIDIA) | TensorRT (`engine`) — exportar **en el propio Jetson** |

In [ ]:
EXPORTS = {
    "onnx":   dict(format="onnx", simplify=True),
    "ncnn":   dict(format="ncnn"),
    "tflite": dict(format="tflite", int8=True, data=str(DATA_YAML)),  # cuantizado INT8
    # "openvino": dict(format="openvino"),
}

exported = {}
for name, kwargs in EXPORTS.items():
    try:
        out = best_model.export(imgsz=IMGSZ, **kwargs)
        exported[name] = out
        print(f"OK {name}: {out}")
    except Exception as e:
        print(f"Error exportando {name}: {e}")

## 12. Copiar el modelo final a `models/`

In [ ]:
import shutil

final_dir = MODELS_DIR / "final"
final_dir.mkdir(exist_ok=True)

shutil.copy2(BEST_PT, final_dir / f"{RUN_NAME}_best.pt")
for name, out in exported.items():
    out = Path(out)
    dst = final_dir / out.name
    if out.is_dir():
        shutil.copytree(out, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(out, dst)

print("Modelos finales en:", final_dir)
for p in final_dir.iterdir():
    print(" -", p.name)